# 01 — From source to searchable chunks

**10–15 minute lab.** Follow three checkpoints: convert one Markdown source, inspect its
structure, then turn it into small index records. The main path is deterministic and needs
neither PostgreSQL nor Ollama.

**Flow:** source → canonical Markdown → AST → chunks → index concept.


In [ ]:
from pathlib import Path

from raglab import SourceInput
from raglab.chunking import ChunkingConfig, SemanticChunker, SimpleTokenCounter
from raglab.conversion import Converter
from raglab.parsing import MarkdownParser

project_root = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
source_path = project_root / "data/samples/aster_greenhouse_controller_manual.md"


## Checkpoint 1 — Objective: create canonical Markdown

**Run:** convert the controlled source through RAGLab's real `Converter` API.


In [ ]:
print("INPUT DOCUMENT")
print(f"  File: {source_path.name}")
print(f"  Location: {source_path.relative_to(project_root)}")
print("  Purpose: short teaching lab")

converted = Converter().convert(SourceInput.path(source_path, purpose="short-lab"))
print("\nCONVERTED CONTENT")
print(f"  Source: {converted.source_name}")
print(f"  Converter: {converted.converter}")
print(f"  Content fingerprint: {converted.content_hash[:12]}…")
print("  Markdown preview:")
for line in converted.markdown.splitlines()[:4]:
    print(f"    {line}")

### What to observe

Expect one stable content hash and a Markdown heading in the preview. Conversion creates a
canonical representation before parsing or chunking.

### Conclusion

A source file is input; canonical Markdown is the normalized document the pipeline can inspect.
URL sources use the same conversion API, but accept only credential-free HTTP(S) targets whose
DNS answers are exclusively public. Every redirect is checked before it is followed; this
hermetic lab deliberately uses a tracked local file instead.


## Checkpoint 2 — Objective: reveal document structure

**Run:** parse the canonical Markdown into typed AST blocks with heading context.


In [ ]:
print("CONVERTED CONTENT")
print(f"  Document title: {converted.source_name}")
print("  Representation: canonical Markdown")

parsed = MarkdownParser().parse(converted)
rows = [
    (block.kind.value, " > ".join(block.heading_path) or "Document root", block.content)
    for block in parsed.blocks
    if block.kind.value != "heading"
]
print("\nPARSED CONTENT")
print(f"  Parsed title: {parsed.title}")
print(f"  Total AST blocks: {len(parsed.blocks)}")
print("  First content blocks:")
for position, (kind, heading, content) in enumerate(rows[:5], start=1):
    preview = " ".join(content.split())[:75]
    print(f"  {position}. Type: {kind} | Section: {heading}")
    print(f"     Text: {preview}")

### What to observe

Expect several block types and heading paths. The AST stores meaning that a plain string does
not: paragraphs know which section contains them.

### Conclusion

Structure supplies boundaries and context before token limits are applied. Parsing must produce
at least one non-empty block or raise `ParsingError`. A document containing only headings is still
valid Markdown, so indexability is enforced by the next stage rather than guessed here.


## Checkpoint 3 — Objective: build the indexable unit

**Run:** chunk the parsed document with the real chunker, then represent each chunk as one
searchable index record.


In [ ]:
chunker = SemanticChunker(
    token_counter=SimpleTokenCounter(),
    config=ChunkingConfig(target_tokens=80, min_tokens=30, max_tokens=120),
)
chunks = chunker.chunk(parsed)
index_records = [
    {
        "position": chunk.index + 1,
        "text": chunk.content,
        "embedding_text": chunk.embedding_text,
        "heading": " > ".join(chunk.heading_path) or "Document root",
    }
    for chunk in chunks
]

print("CHUNKS")
print(f"  Created: {len(index_records)} searchable chunks")
for record in index_records[:3]:
    preview = " ".join(record["text"].split())[:80]
    print(f"  {record['position']}. Section: {record['heading']}")
    print(f"     Evidence: {preview}")

query = "irrigation flow"
query_terms = set(query.split())
matches = [
    record
    for record in index_records
    if query_terms & set(record["embedding_text"].lower().split())
]
print("\nINDEXING RESULT")
print(f"  Records ready to index: {len(index_records)}")
print(f"  Demonstration query: {query!r}")
print(f"  Readable matches: {len(matches)}")
for rank, record in enumerate(matches[:3], start=1):
    preview = " ".join(record["text"].split())[:80]
    print(f"  Rank {rank} | Section: {record['heading']}")
    print(f"    Evidence: {preview}")

### What to observe

Expect multiple compact records and at least one irrigation-related match. `min_tokens` is a
preferred size: maximum-size, heading, and semantic cuts remain hard boundaries, while only a
small target-size tail may merge backward within the same heading and `max_tokens`. A production
index adds embeddings and BM25 statistics, but its core job is the same: map a query to chunk IDs.

### Conclusion

The index does not store a magical answer. It makes evidence chunks findable while preserving
their text and structural context. `SemanticChunker` raises `ChunkingError` when parsed Markdown
cannot produce a searchable chunk—for example, when it contains headings but no body content.
The ingestion pipeline repeats that guard before idempotency lookup, final embeddings, or storage,
so a non-searchable document cannot be reported as indexed.

## Optional appendix — real services and deeper experiments

Use `raglab-ingest` for Ollama + PostgreSQL indexing, and use
`benchmark_ingestion_hyperparameters.ipynb` for chunking/HNSW experiments. They are deliberately
outside this short learning path.
